# 07b · 스캔 PDF 글자 복구

08에서 글자 수가 부족하다고 나온 쪽을 한국어 OCR로 읽는다. PDF는 손대지 않고, 원문 PDF 페이지 번호와 OCR 신뢰도를 기록해 `data/personal/corpus/ocr_pages/`에 둔다.


In [ ]:
from pathlib import Path
import json, sys
from IPython.display import display
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None, 'SANUP-P 프로젝트 폴더에서 실행하세요.'
sys.path.insert(0,str(ROOT))
from src.ocr_recovery import recover


In [ ]:
summary=recover(ROOT)
display(summary)


In [ ]:
records=[json.loads(p.read_text(encoding='utf-8')) for p in (ROOT/'data/personal/corpus/ocr_pages').glob('*.json')]
accepted=[r for r in records if r['accepted_for_index']]
print('OCR 기록',len(records),'페이지 / 색인 가능',len(accepted),'페이지')
print('평균 OCR 신뢰도',round(sum(r['mean_accepted_line_score'] for r in accepted)/max(len(accepted),1),3))
display([{k:r[k] for k in ['doc_id','page','chars','mean_accepted_line_score','accepted_for_index']} for r in accepted[:12]])


## OCR 보완 결과와 검토 대상

- 텍스트 추출량이 부족했던 PDF 105쪽을 한국어 OCR로 처리했다. 99쪽은 80자 이상 텍스트를 확보해 청킹 대상으로 돌렸고 6쪽은 OCR 결과가 검색 근거로 쓰기 부족해 제외했다.
- 표지 표본의 평균 OCR 신뢰도는 0.977이지만 표지 점수만으로 본문 인식 정확도를 보증할 수 없다. 문서 유형별·본문 페이지별 대조가 필요하다.
- OCR 문장은 원본 PDF 페이지로 돌아갈 수 있게 문서 ID·쪽·원문 URL과 `ocr_review_required` 표시를 유지한다. 앱에서 해당 표시를 사용자에게 보인다.

### 사용할 때 주의할 점

- 자동 인식된 숫자·단위·부정어는 안전 기준을 바꿀 수 있으므로 인용 전에 원본 페이지와 대조한다.
- 6쪽 제외로 해당 내용은 검색되지 않을 수 있다. 관련 질문에 답이 없다고 해서 공식 문서에 내용이 없다고 결론 내리지 않는다.
- OCR은 원본 문서를 고친 것이 아니라 검색용 텍스트를 보완한 것이다. 원본 PDF는 그대로 보존한다.
